# EyeRefine_Train — 2단계 눈 정밀 모델 (눈 8점 + 눈썹 9점 + 눈동자 1점)

**왜 필요한가**
`WFLW_Train.ipynb` 의 1단계 모델(MobileNetV2 → GAP → Dense 회귀)은 얼굴 전체 위치는 잘 잡지만, 눈·눈썹처럼 작은 부위는
학습 데이터의 *평균 모양* 으로 수렴한다. 그래서 눈꼬리가 올라갔는지 내려갔는지가 측정되지 않는다 (Celeb‑K 기준표에서
`eye_len` 변동계수 1.3%, `brow_len` 0.5% 가 그 증거).

**만드는 것**
`eye_refine.tflite` — 1단계 결과로 눈+눈썹 주변을 64×64 로 잘라 넣으면 18점(눈 8 + 눈썹 9 + 눈동자 1)을 다시 찍는 작은 CNN.
오른쪽 눈은 좌우 반전해 '왼쪽 눈'으로 넣으므로 모델 하나로 양쪽을 처리한다 (크롭 규칙은 `eye_prep.py`, 보드와 공용).

**미리 준비할 것** — `WFLW_Train.ipynb` 와 같음 (WFLW tar.gz 두 개, 드라이브의 `FaceReading_3/` 폴더에 `wflw_prep.py`, `eye_prep.py`).
1단계 비교는 `files/save/wflw_mesh.tflite` (WFLW_Train 결과) 를 자동으로 찾는다. 런타임 GPU(T4), 전처리 5분 + 학습 15분 정도.


## 1. 드라이브 마운트 · 경로

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    colab = True
except Exception:
    colab = False
print('colab' if colab else 'local')

In [ ]:
import os, sys
if colab:
    BASE     = '/content/drive/MyDrive/ai_semiconductor/Python/examples/'
    CODE_DIR = BASE + 'proj_face/FaceReading_3/'          # wflw_prep.py, eye_prep.py 가 있는 곳
    DATA_DIR = BASE + 'files/WFLW/'
    save_dir = BASE + 'files/save/'
    work_dir = '/content/wflw'
else:
    CODE_DIR = './'; DATA_DIR = '../files/WFLW/'; save_dir = '../files/save/'; work_dir = './wflw'
os.makedirs(save_dir, exist_ok=True); os.makedirs(work_dir, exist_ok=True)
sys.path.insert(0, CODE_DIR)
from wflw_prep import IMG, crop_box, crop_face, to_input, pts_to_image
from eye_prep import (EYE_IMG, EYE_MARGIN, EYE_IDS, N_EYE_PTS, eye_crop_box, crop_eye, eye_to_input,
                      eye_pts_to_crop, eye_pts_to_image)
print('EYE_IMG =', EYE_IMG, ' EYE_MARGIN =', EYE_MARGIN, ' points =', N_EYE_PTS)

## 2. 압축 해제 · 라벨 읽기

In [ ]:
!tar -xzf "{DATA_DIR}WFLW_images.tar.gz" -C {work_dir}
!tar -xzf "{DATA_DIR}WFLW_annotations.tar.gz" -C {work_dir}
import glob
IMG_ROOT = glob.glob(work_dir + '/**/WFLW_images', recursive=True)[0] + '/'
ANN_DIR  = glob.glob(work_dir + '/**/list_98pt_rect_attr_train_test', recursive=True)[0] + '/'
print(IMG_ROOT, ANN_DIR)

눈 크롭은 **원본 해상도**에서 잘라야 한다 (160px 얼굴 크롭에서는 눈이 20px 밖에 안 됨).
학습 때는 1단계 모델의 위치 오차를 흉내 내야 하므로, 정답 점으로 잡은 정사각을 조금 넓게(`STORE_MARGIN`) 96px 로 저장해 두고
증강 단계에서 그 안을 무작위로 이동·확대·회전해서 64px 입력을 만든다.

In [ ]:
import numpy as np, cv2, time
STORE = 96                                   # 저장 크롭 한 변
STORE_MARGIN = EYE_MARGIN * 1.4              # 증강 여유를 위해 넓게 저장
NOMINAL = STORE * EYE_MARGIN / STORE_MARGIN  # 저장 크롭 안에서 '정상 크롭' 이 차지하는 한 변 (px)

def load_eyes(txt, root, keep_meta=False):
    """라벨 txt → X (N,96,96,3) uint8 눈 크롭 (왼쪽 눈 모양으로 통일), Y (N,18,2) 크롭 픽셀 좌표, meta"""
    X, Y, meta = [], [], []
    t0 = time.time()
    for k, line in enumerate(open(txt)):
        t = line.split()
        if len(t) != 207: continue
        pts = np.array(t[:196], np.float32).reshape(98, 2)
        img = cv2.imread(root + t[-1])
        if img is None: continue
        for side in ('left', 'right'):
            x0, y0, s = eye_crop_box(pts, side, STORE_MARGIN)
            pad = int(s) + 10
            p = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0, 0, 0))
            x0i, y0i, si = int(round(x0)) + pad, int(round(y0)) + pad, max(int(round(s)), 1)
            crop = cv2.resize(p[y0i:y0i + si, x0i:x0i + si], (STORE, STORE), interpolation=cv2.INTER_AREA)
            q = eye_pts_to_crop(pts, x0, y0, s, side) * STORE          # 0~1 → 저장 크롭 픽셀 (오른쪽은 이미 x 반전됨)
            if side == 'right': crop = crop[:, ::-1].copy()
            X.append(crop); Y.append(q)
        if keep_meta:
            meta.append((t[-1], pts, tuple(map(float, t[196:200]))))
        if k % 1000 == 0: print(k, '%.0fs' % (time.time() - t0))
    return np.array(X, np.uint8), np.array(Y, np.float32), meta

cache = work_dir + '/wflw_eyes_%d.npz' % STORE
if os.path.exists(cache):
    d = np.load(cache, allow_pickle=True); Xtr, Ytr, Xte, Yte = d['Xtr'], d['Ytr'], d['Xte'], d['Yte']; meta_te = list(d['meta'])
else:
    Xtr, Ytr, _ = load_eyes(ANN_DIR + 'list_98pt_rect_attr_train.txt', IMG_ROOT)
    Xte, Yte, meta_te = load_eyes(ANN_DIR + 'list_98pt_rect_attr_test.txt', IMG_ROOT, keep_meta=True)
    np.savez_compressed(cache, Xtr=Xtr, Ytr=Ytr, Xte=Xte, Yte=Yte, meta=np.array(meta_te, dtype=object))
print(Xtr.shape, Ytr.shape, Xte.shape, Yte.shape)   # (15000,96,96,3) (15000,18,2) (5000,...)

In [ ]:
import matplotlib.pyplot as plt
def show_eyes(imgs, ptss, cols=8, title=None, pred=None, size=None):
    size = size or imgs.shape[1]
    plt.figure(figsize=(1.6 * cols, 1.9))
    for i in range(cols):
        im = cv2.resize(imgs[i], (128, 128), interpolation=cv2.INTER_NEAREST); f = 128.0 / size
        for x, y in ptss[i].reshape(-1, 2): cv2.circle(im, (int(x * f), int(y * f)), 2, (0, 255, 0), -1)
        if pred is not None:
            for x, y in pred[i].reshape(-1, 2): cv2.circle(im, (int(x * f), int(y * f)), 2, (0, 0, 255), -1)
        plt.subplot(1, cols, i + 1); plt.imshow(im[:, :, ::-1]); plt.axis('off')
    if title: plt.suptitle(title)
    plt.show()
show_eyes(Xtr, Ytr, title='stored 96px crops (green = label). 오른쪽 눈도 왼쪽 모양으로 반전되어 있어야 함')

## 3. 증강 — 1단계 모델의 오차를 흉내 낸다
저장 크롭(96px, 넓은 여유) 안에서 중심 이동 ±12%, 크기 0.8~1.25배, 회전 ±20°, 밝기 변화를 주고 64px 로 뽑는다.
증강이 없으면(`jitter=False`) 정확히 `EYE_MARGIN` 크롭과 같아진다 → 보드에서 1단계 점으로 자른 것과 동일.

In [ ]:
def eye_affine(img, pts, jitter=True):
    """저장 크롭 (96,96,3) + 점 (18,2 px) → (EYE_IMG,EYE_IMG,3), (18,2) 0~1"""
    c = STORE / 2.0
    if jitter:
        sc = np.random.uniform(0.8, 1.25); ang = np.random.uniform(-20, 20)
        dx, dy = np.random.uniform(-0.12, 0.12, 2) * NOMINAL
    else:
        sc, ang, dx, dy = 1.0, 0.0, 0.0, 0.0
    k = EYE_IMG / (NOMINAL * sc)                               # 저장 px → 출력 px
    M = cv2.getRotationMatrix2D((c + dx, c + dy), ang, k)
    M[:, 2] += (EYE_IMG / 2.0 - (c + dx), EYE_IMG / 2.0 - (c + dy))
    out = cv2.warpAffine(img, M, (EYE_IMG, EYE_IMG), borderMode=cv2.BORDER_REPLICATE)
    q = (np.c_[pts, np.ones(len(pts))] @ M.T) / EYE_IMG
    if jitter:
        out = np.clip(out.astype(np.float32) * np.random.uniform(0.7, 1.3) + np.random.uniform(-25, 25), 0, 255).astype(np.uint8)
        if np.random.rand() < 0.3:                              # 가벼운 블러 (저해상도 웹캠 흉내)
            out = cv2.GaussianBlur(out, (3, 3), 0)
    return out, q.astype(np.float32)

def gen(X, Y, bs=128):
    n = len(X)
    while True:
        idx = np.random.permutation(n)
        for i in range(0, n - bs + 1, bs):
            xb, yb = [], []
            for j in idx[i:i + bs]:
                im, q = eye_affine(X[j], Y[j]); xb.append(im); yb.append(q.reshape(-1))
            yield np.array(xb, np.float32)[:, :, :, ::-1] / 255., np.array(yb, np.float32)

def to_eval(X, Y):
    xs, ys = zip(*[eye_affine(X[j], Y[j], jitter=False) for j in range(len(X))])
    return np.array(xs, np.float32)[:, :, :, ::-1] / 255., np.array(ys, np.float32).reshape(len(X), -1)

xb, yb = next(gen(Xtr, Ytr, 8))
show_eyes((xb[:, :, :, ::-1] * 255).astype(np.uint8), yb * EYE_IMG, title='augmented 64px (초록 점이 눈·눈썹에 맞아야 함)')
Xte_in, Yte_in = to_eval(Xte, Yte)
print(Xte_in.shape, Yte_in.shape)

## 4. 모델 — 작은 CNN, GAP 없이 Flatten (공간 정보를 버리지 않는다)
약 0.6M 파라미터, 양자화 후 ~0.6MB. 64×64 입력이라 라즈베리파이에서 눈 두 개 합쳐 수 ms.

In [ ]:
import tensorflow as tf
L = tf.keras.layers
def block(x, ch):
    x = L.SeparableConv2D(ch, 3, padding='same', use_bias=False)(x); x = L.BatchNormalization()(x); x = L.ReLU()(x)
    x = L.SeparableConv2D(ch, 3, padding='same', use_bias=False)(x); x = L.BatchNormalization()(x); x = L.ReLU()(x)
    return L.MaxPool2D()(x)

inp = L.Input((EYE_IMG, EYE_IMG, 3))
x = L.Conv2D(24, 3, padding='same', use_bias=False)(inp); x = L.BatchNormalization()(x); x = L.ReLU()(x)
x = block(x, 32)      # 32
x = block(x, 64)      # 16
x = block(x, 96)      # 8
x = block(x, 128)     # 4
x = L.Flatten()(x)                                   # 4*4*128 = 2048  (GAP 대신 Flatten: 위치 정보 유지)
x = L.Dense(256, activation='relu')(x); x = L.Dropout(0.2)(x)
out = L.Dense(N_EYE_PTS * 2, activation='sigmoid')(x)
model = tf.keras.Model(inp, out)

def wing_loss(y, p, w=0.1, eps=0.02):
    d = tf.abs(y - p); C = w - w * tf.math.log(1 + w / eps)
    return tf.reduce_mean(tf.where(d < w, w * tf.math.log(1 + d / eps), d - C))

model.compile(tf.keras.optimizers.Adam(1e-3), loss=wing_loss)
model.summary()

In [ ]:
%%time
EPOCHS = 50
hist = model.fit(gen(Xtr, Ytr), steps_per_epoch=len(Xtr) // 128, epochs=EPOCHS,
                 validation_data=(Xte_in, Yte_in),
                 callbacks=[tf.keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5, verbose=1),
                            tf.keras.callbacks.ModelCheckpoint(work_dir + '/eye_best.keras', save_best_only=True)])

## 5. 평가 — 눈 부위 NME · 눈꼬리 각도가 정답을 따라가는지

In [ ]:
model = tf.keras.models.load_model(work_dir + '/eye_best.keras', custom_objects={'wing_loss': wing_loss})
Pr = model.predict(Xte_in, verbose=0).reshape(-1, N_EYE_PTS, 2); G = Yte_in.reshape(-1, N_EYE_PTS, 2)
# 18점 순서: 0~8 눈썹(33~41), 9~16 눈(60~67: 9=바깥꼬리, 13=안쪽꼬리), 17 눈동자
eye_w = np.linalg.norm(G[:, 9] - G[:, 13], axis=1)
nme_eye = (np.linalg.norm(Pr[:, 9:17] - G[:, 9:17], axis=2).mean(1) / eye_w).mean() * 100
nme_brow = (np.linalg.norm(Pr[:, :9] - G[:, :9], axis=2).mean(1) / eye_w).mean() * 100
print('refine 모델  눈 NME = %.2f %%   눈썹 NME = %.2f %%   (눈 폭 기준)' % (nme_eye, nme_brow))

def tilt(p):   # 안쪽 꼬리 → 바깥 꼬리 기울기(°). 위로 올라가면 +  (face_metrics._tilt_deg 와 같은 정의, 왼쪽 눈 기준)
    inner, outer = p[:, 13], p[:, 9]
    return np.degrees(np.arctan2(-(outer[:, 1] - inner[:, 1]), np.abs(outer[:, 0] - inner[:, 0])))
tg, tp = tilt(G), tilt(Pr)
print('눈꼬리 각도: 정답 std %.2f°,  예측 MAE %.2f°,  상관 %.3f' % (tg.std(), np.abs(tg - tp).mean(), np.corrcoef(tg, tp)[0, 1]))

plt.figure(figsize=(6, 4)); plt.plot(hist.history['loss'], label='train'); plt.plot(hist.history['val_loss'], label='val')
plt.title('Eye refine  (eye NME %.2f%%)' % nme_eye); plt.xlabel('epoch'); plt.ylabel('wing loss'); plt.grid(True); plt.legend()
plt.savefig(save_dir + 'eye_train_log.png', dpi=150, bbox_inches='tight'); plt.show()
idx = np.random.choice(len(Xte), 8, replace=False)
show_eyes((Xte_in[idx][:, :, :, ::-1] * 255).astype(np.uint8), G[idx] * EYE_IMG, pred=Pr[idx] * EYE_IMG, title='test: green = label, red = refine')
plt.savefig(save_dir + 'eye_samples.png', dpi=150, bbox_inches='tight')

### 5-1. 1단계(wflw_mesh)만 썼을 때 vs 2단계까지 썼을 때 — 보드와 같은 경로로 비교
`FaceReading_3/wflw_mesh.tflite` 또는 `files/save/wflw_mesh.tflite` 가 있으면: 원본 사진 → 얼굴 크롭 → 1단계 98점 → 그 점으로 눈 크롭 → 2단계 18점.
이 수치가 PPT 에 들어갈 "왜 2단계가 필요한가" 의 근거다.

In [ ]:
mesh_path = next((p for p in (CODE_DIR + 'wflw_mesh.tflite', save_dir + 'wflw_mesh.tflite') if os.path.exists(p)), None)
if mesh_path:
    it = tf.lite.Interpreter(model_path=mesh_path); it.allocate_tensors()
    mi, mo = it.get_input_details()[0], it.get_output_details()[0]
    rows = []                                  # (gt_tilt, coarse_tilt, refined_tilt, side)
    coarse_err, refined_err = [], []
    for path, gt, box in meta_te[:400]:
        img = cv2.imread(IMG_ROOT + path)
        x0, y0, s = crop_box(*box)
        it.set_tensor(mi['index'], to_input(crop_face(img, x0, y0, s))); it.invoke()
        coarse = pts_to_image(it.get_tensor(mo['index'])[0].reshape(98, 2), x0, y0, s)
        for side in ('left', 'right'):
            ex0, ey0, es = eye_crop_box(coarse, side)                      # 보드와 같은 2단계 크롭
            q = model.predict(eye_to_input(crop_eye(img, ex0, ey0, es, side)), verbose=0)[0]
            refined = eye_pts_to_image(q, ex0, ey0, es, side)
            ids = EYE_IDS[side]
            ew = np.linalg.norm(gt[ids[9]] - gt[ids[13]])
            coarse_err.append(np.linalg.norm(coarse[ids] - gt[ids], axis=1).mean() / ew)
            refined_err.append(np.linalg.norm(refined - gt[ids], axis=1).mean() / ew)
            def t(p):   # 안쪽→바깥쪽, 양쪽 모두 '올라감 +'
                inner, outer = p[ids[13]], p[ids[9]]
                return np.degrees(np.arctan2(-(outer[1] - inner[1]), abs(outer[0] - inner[0])))
            rows.append((t(gt), t(coarse), t(refined)))
    r = np.array(rows)
    print('눈+눈썹 18점 오차 (눈 폭 기준 NME):  1단계만 %.1f%%  →  2단계 %.1f%%' % (np.mean(coarse_err) * 100, np.mean(refined_err) * 100))
    print('눈꼬리 각도 (정답 std %.2f°)' % r[:, 0].std())
    print('   1단계만 : 예측 std %.2f°  MAE %.2f°  상관 %.3f' % (r[:, 1].std(), np.abs(r[:, 0] - r[:, 1]).mean(), np.corrcoef(r[:, 0], r[:, 1])[0, 1]))
    print('   2단계   : 예측 std %.2f°  MAE %.2f°  상관 %.3f' % (r[:, 2].std(), np.abs(r[:, 0] - r[:, 2]).mean(), np.corrcoef(r[:, 0], r[:, 2])[0, 1]))
    plt.figure(figsize=(8, 3.6))
    for i, name in ((1, '1-stage only'), (2, '1-stage + eye refine')):
        plt.subplot(1, 2, i); plt.scatter(r[:, 0], r[:, i], s=6, alpha=.5); plt.plot([-15, 25], [-15, 25], 'r--', lw=1)
        plt.xlabel('ground-truth eye tilt (deg)'); plt.ylabel('predicted'); plt.title(name); plt.grid(True)
    plt.tight_layout(); plt.savefig(save_dir + 'eye_tilt_compare.png', dpi=150, bbox_inches='tight'); plt.show()
else:
    print('wflw_mesh.tflite 가 FaceReading_3/ 에도 files/save/ 에도 없어 1단계 비교를 건너뜀')

## 6. TFLite 변환 · 검증 · 저장

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tfl = conv.convert()
open(save_dir + 'eye_refine.tflite', 'wb').write(tfl)
print('eye_refine.tflite  %.1f KB' % (len(tfl) / 1024))
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
i, o = it.get_input_details()[0], it.get_output_details()[0]
errs = []
for k in range(50):
    it.set_tensor(i['index'], Xte_in[k:k + 1]); it.invoke()
    errs.append(np.abs(it.get_tensor(o['index'])[0] - Pr[k].reshape(-1)).max() * EYE_IMG)
print('keras vs tflite 최대 차이: %.2f px (64px 기준)' % max(errs))
for f in ['eye_refine.tflite', 'eye_train_log.png', 'eye_samples.png', 'eye_tilt_compare.png']:
    print('%-26s %s' % (f, 'OK' if os.path.exists(save_dir + f) else '없음'))

## 7. 다음 단계
1. `save/eye_refine.tflite` 를 보드 `FRD/` 폴더(face_fortune.py 옆)에 복사 → 실행하면 자동으로 2단계가 켜진다 (`--no-eye-refine` 로 끌 수 있음).
2. 눈·눈썹 관련 기준표 항목(`eye_len*`, `eye_h_*`, `eye_aspect`, `eye_gap`, `eye_tilt`, `under_eye`, `brow_*`, `glabella_w`)은
   2단계를 켠 상태로 Celeb‑K 를 다시 돌려 `face_reference_10level_clean.csv` 를 갱신해야 한다 (다른 항목은 그대로).
3. `eye_tilt_compare.png` 를 PPT 에 넣으면 "1단계만으로는 눈꼬리가 평균으로 뭉개진다 → 2단계로 해결" 을 한 장으로 설명할 수 있다.
